# Fase 2 - Comprensión de los Datos
## Sección 08: Variables Categóricas & Sección 09: Matriz de Correlaciones

**Notebook:** notebooks/08_EDA_variables_categoricas.ipynb
**Responsable:** Sofia | **Apoyo:** Steve
**Objetivo:** Analizar distribucion de tipos de propiedad, cobertura temporal/geográfica, y correlaciones entre variables numéricas.

## Configuración inicial

In [ ]:
from config import *

---
## Sección 08: Variables Categóricas
## Tipo de propiedad

### Cargar datasets con tipo de propiedad

In [ ]:
TYPE_CFG = {
    'A1': ('A1_colombia_housing_properties.csv', 'property_type'),
    'A2': ('A2_fincaraiz_colombia.csv', 'Tipo Propiedad'),
    'A5': ('A5_medellin_properties_2023.csv', 'property_type'),
    'A6': ('A6_real_estate_bogota_2023.csv', 'tipo_de_inmueble'),
    'A7': ('A7_fincaraiz_villavicencio_scraping.csv', 'tipo_inmueble'),
}

type_data = []
for fid, (fname, tcol) in TYPE_CFG.items():
    df = pd.read_csv(os.path.join(RAW, fname), encoding='utf-8-sig', low_memory=False)
    if tcol in df.columns:
        temp = df[[tcol]].copy()
        temp['tipo'] = temp[tcol].astype(str).str.strip().str.lower()
        temp['fuente'] = fid
        type_data.append(temp[['tipo','fuente']])
        print(f'{fid}: {df.shape[0]:>8,} filas  columna={tcol}')

df_tipos = pd.concat(type_data, ignore_index=True)
print(f'\nTotal registros con tipo: {len(df_tipos):,}')

### Distribución de tipos de propiedad (top 15)

In [ ]:
type_dist = df_tipos['tipo'].value_counts()
type_pct = df_tipos['tipo'].value_counts(normalize=True) * 100
print('--- DISTRIBUCION DE TIPOS DE PROPIEDAD ---')
print(f"{'Tipo':30s} {'N':>10s} {'%':>8s}")
for tipo in type_dist.index:
    print(f'{tipo:30s} {type_dist[tipo]:>10,} {type_pct[tipo]:>7.2f}%')
print(f'\nTipos unicos: {len(type_dist)}')

### Gráfico de barras top 8 tipos

In [ ]:
top8 = type_dist.head(8)
plt.figure(figsize=(12, 6))
colors = sns.color_palette('Set2', len(top8))
bars = plt.bar(range(len(top8)), top8.values, color=colors, edgecolor='white')
plt.xticks(range(len(top8)), top8.index, rotation=45, ha='right')
for bar, val in zip(bars, top8.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(top8.values)*0.01, f'{val:,}\n({val/len(df_tipos)*100:.1f}%)', ha='center', fontsize=9)
plt.ylabel('Frecuencia')
plt.title('Top 8 tipos de propiedad mas frecuentes — Apartamento y Casa dominan (~80%)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

fig.suptitle("Top 8 tipos de propiedad mas frecuentes — Apartamento y Casa dominan el mercado (~80% del total)", fontsize=14, y=1.02)
plt.savefig(os.path.join(FIGS, 'top8_tipos_propiedad.png'), dpi=150, bbox_inches='tight')

print("Grafico guardado en: top8_tipos_propiedad.png")
plt.show()


**Conclusion tipos de propiedad:**
- Apartamento y Casa concentran ~80% del mercado — el resto son tipos minoritarios (<1% c/u).
- Los tipos minoritarios podrian agruparse en una categoria "Otros" para Fase 3.
- La distribucion por fuente revela sesgos: A1 (Properati) tiene mayor diversidad de tipos.


### Tipos minoritarios

In [ ]:
minor_types = type_dist[type_dist < len(df_tipos) * 0.01]
print('--- TIPOS MINORITARIOS (<1% del total) ---')
print(f'Cantidad de tipos minoritarios: {len(minor_types)}')
print(f'Registros en tipos minoritarios: {minor_types.sum():,} ({minor_types.sum()/len(df_tipos)*100:.2f}% del total)\n')
for tipo, count in minor_types.items():
    print(f'  {tipo:35s} {count:>8,} ({count/len(df_tipos)*100:.3f}%)')

### Distribución por tipo y fuente

In [ ]:
cross_type = pd.crosstab(df_tipos['tipo'], df_tipos['fuente'])
cross_type['Total'] = cross_type.sum(axis=1)
cross_type = cross_type.sort_values('Total', ascending=False)
print('--- TIPO DE PROPIEDAD POR FUENTE ---')
display(cross_type.head(12))

cross_pct = cross_type.div(cross_type.sum(axis=0), axis=1) * 100
print('\nDistribucion porcentual:')
display(cross_pct.head(12).round(1))

---
## Volumen temporal y geográfico

### Cargar datos para cobertura temporal

In [ ]:
TEMP_CFG = {
    'A1': ('A1_colombia_housing_properties.csv', 'created_on', None),
    'A2': ('A2_fincaraiz_colombia.csv', 'Fecha Actualizacion', 'Ciudad'),
    'A5': ('A5_medellin_properties_2023.csv', None, None),
    'A7': ('A7_fincaraiz_villavicencio_scraping.csv', None, 'ciudad'),
}

temp_data = []
for fid, (fname, dcol, ccol) in TEMP_CFG.items():
    df = pd.read_csv(os.path.join(RAW, fname), encoding='utf-8-sig', low_memory=False)
    if dcol and dcol in df.columns:
        df[dcol] = pd.to_datetime(df[dcol], errors='coerce')
        df['year'] = df[dcol].dt.year
    elif fid == 'A5':
        df['year'] = 2023
    elif fid == 'A7':
        for c in ['fecha_publicacion', 'created_on']:
            if c in df.columns:
                df[c] = pd.to_datetime(df[c], errors='coerce')
                df['year'] = df[c].dt.year
                break
        else:
            df['year'] = None
    else:
        df['year'] = None
    df['fuente'] = fid
    if ccol and ccol in df.columns:
        df['ciudad'] = df[ccol].astype(str).str.strip().str.title()
    else:
        df['ciudad'] = fid
    temp_data.append(df[['year','fuente','ciudad']])
    print(f'{fid}: {df.shape[0]:>8,} filas')

df_temp = pd.concat(temp_data, ignore_index=True)
df_temp = df_temp[df_temp['year'].between(2018, 2024, inclusive='both')]
print(f'\nTotal registros con year 2018-2024: {len(df_temp):,}')

### Registros por year

In [ ]:
yearly = df_temp['year'].value_counts().sort_index()
print('--- REGISTROS POR YEAR ---')
for y, v in yearly.items():
    print(f'  {int(y)}: {v:>8,}')

plt.figure(figsize=(10, 5))
bars = plt.bar(yearly.index.astype(int), yearly.values, color='steelblue', edgecolor='white')
for bar in bars:
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+100, f'{int(bar.get_height()):,}', ha='center', fontsize=9)
plt.xlabel('Year')
plt.ylabel('Registros')
plt.title('Volumen de registros por year (2018-2024) — mayoria de datos en 2020-2024')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

fig.suptitle("Volumen de registros por year (2018-2024) — la mayoria de los datos se concentran en 2020-2024", fontsize=14, y=1.02)
plt.savefig(os.path.join(FIGS, 'registros_por_year.png'), dpi=150, bbox_inches='tight')

print("Grafico guardado en: registros_por_year.png")
plt.show()


**Conclusion registros por year:**
- La mayoria de los registros se concentran en 2020-2024, con pico en 2022-2023.
- Years anteriores (2018-2019) tienen menor cobertura — posible sesgo temporal.
- Para analisis longitudinal, considerar ponderar por volumen de registros por year.


### Registros por ciudad top 12

In [ ]:
city_vol = df_temp['ciudad'].value_counts().head(12)
print('--- TOP 12 CIUDADES POR VOLUMEN ---')
for c, v in city_vol.items():
    print(f'  {c:30s} {v:>8,} ({v/len(df_temp)*100:5.1f}%)')

plt.figure(figsize=(12, 7))
colors = sns.color_palette('viridis', len(city_vol))
bars = plt.barh(range(len(city_vol)), city_vol.values, color=colors, edgecolor='white')
plt.yticks(range(len(city_vol)), city_vol.index)
for i, (c, v) in enumerate(city_vol.items()):
    plt.text(v + max(city_vol.values)*0.005, i, f'{v:,} ({v/len(df_temp)*100:.1f}%)', va='center', fontsize=9)
plt.xlabel('Registros')
plt.title('Top 12 ciudades por volumen de registros — Bogota y Medellin concentran la mayor parte')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()

fig.suptitle("Top 12 ciudades por volumen de registros — Bogota y Medellin concentran la mayor parte de los datos", fontsize=14, y=1.02)
plt.savefig(os.path.join(FIGS, 'top12_ciudades_volumen.png'), dpi=150, bbox_inches='tight')

print("Grafico guardado en: top12_ciudades_volumen.png")
plt.show()


**Conclusion cobertura geografica:**
- Bogota y Medellin concentran la mayoria del volumen de datos.
- Ciudades intermedias como Ibague, Cucuta, Pereira tienen cobertura insuficiente (<80% de years).
- Para Fase 3, considerar si estas ciudades requieren fuentes adicionales de datos.


### Tabla cruzada ciudad x year

In [ ]:
cross_cy = pd.crosstab(df_temp['ciudad'], df_temp['year'])
cross_cy = cross_cy[cross_cy.sum(axis=1) >= 100]
print(f'Ciudades con >=100 registros: {len(cross_cy)}')
print('\n--- TABLA CRUZADA CIUDAD x YEAR ---')
display(cross_cy)

print('\nCobertura (% de years con datos):')
coverage = (cross_cy > 0).sum(axis=1) / cross_cy.shape[1] * 100
for ciudad, pct in coverage.sort_values(ascending=False).items():
    bars = '#' * int(pct / 5)
    print(f'  {ciudad:30s} {pct:5.0f}%  {bars}')

### Ciudades con cobertura insuficiente

In [ ]:
low_cov = coverage[coverage < 80].sort_values()
print('--- CIUDADES CON COBERTURA INSUFICIENTE (<80% de years) ---')
if len(low_cov) > 0:
    for ciudad, pct in low_cov.items():
        years_present = cross_cy.columns[cross_cy.loc[ciudad] > 0].tolist()
        print(f'  {ciudad:30s} {pct:.0f}% cobertura  years: {years_present}')
else:
    print('  Todas las ciudades principales tienen >=80% cobertura.')

print('\nPeriodos con baja cobertura:')
period_low = (cross_cy.sum(axis=0) < cross_cy.sum(axis=0).median() * 0.5)
for y, low in period_low.items():
    if low:
        print(f'  {int(y)}: solo {cross_cy[y].sum():,} registros (bajo)')

---
## Sección 09: Matriz de Correlaciones
## Correlaciónes de variables numéricas

### Cargar datasets con variables numéricas

In [ ]:
NUM_CFG = {
    'A1': ('A1_colombia_housing_properties.csv', {'price':'precio','bedrooms':'rooms','bathrooms':'bathrooms'}, None),
    'A2': ('A2_fincaraiz_colombia.csv', {'Precio':'precio','Habitaciones':'rooms','Banos':'bathrooms','Garages':'parking','Area Construida':'area','Estrato':'estrato'}, 'Ciudad'),
    'A5': ('A5_medellin_properties_2023.csv', {'price':'precio','rooms':'rooms','baths':'bathrooms','garages':'parking','area':'area','stratum':'estrato'}, None),
    'A6': ('A6_real_estate_bogota_2023.csv', {'precio':'precio','habitaciones':'rooms','banos':'bathrooms','garajes':'parking','area':'area','estrato':'estrato'}, None),
    'A7': ('A7_fincaraiz_villavicencio_scraping.csv', {'precio_cop':'precio','habitaciones':'rooms','banos':'bathrooms','parqueaderos':'parking','area_m2':'area','estrato':'estrato'}, None),
}

num_dfs = []
for fid, (fname, cols, ccol) in NUM_CFG.items():
    df = pd.read_csv(os.path.join(RAW, fname), encoding='utf-8-sig', low_memory=False)
    import unicodedata
    def _norm(s): return unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode('ascii').lower().replace(' ', '_')
    df.columns = [_norm(c) for c in df.columns]
    norm_keys = {_norm(k): v for k, v in cols.items()}
    rename = {v: k for k, v in norm_keys.items()}
    sub = df[list(norm_keys.keys())].copy()
    sub = sub.rename(columns=rename)
    for c in ['precio','area','rooms','bathrooms','parking','estrato']:
        if c not in sub.columns:
            sub[c] = None
    sub['fuente'] = fid
    num_dfs.append(sub)
    print(f'{fid}: {len(sub):,} filas')

df_num = pd.concat(num_dfs, ignore_index=True)
print(f'\nTotal: {len(df_num):,} registros')

# Convertir a numeric
for c in ['precio','area','rooms','bathrooms','parking','estrato']:
    df_num[c] = pd.to_numeric(df_num[c], errors='coerce')

print('\nRegistros con todas las variables:')
print(f'  {df_num.dropna(subset=["precio","area","rooms","bathrooms"]).shape[0]:,}')

### Matriz de correlacion

In [ ]:
corr_vars = ['precio', 'area', 'rooms', 'bathrooms', 'parking', 'estrato']
df_corr = df_num[corr_vars].dropna()
print(f'Registros para correlacion: {len(df_corr):,}')

corr_matrix = df_corr.corr(method='spearman')
print('\n--- MATRIZ DE CORRELACION (Spearman) ---')
display(corr_matrix.round(4))

### Heatmap triangular de correlaciones

In [ ]:
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r', vmin=-1, vmax=1,
            square=True, cbar_kws={'label': 'Correlacion Spearman'})
plt.title('Matriz de correlaciones (Spearman) — area tiene la mayor correlacion con precio')
plt.tight_layout()

fig.suptitle("Matriz de correlaciones (Spearman) entre variables numericas — tonos mas oscuros = correlacion mas fuerte", fontsize=14, y=1.02)
plt.savefig(os.path.join(FIGS, 'heatmap_correlaciones.png'), dpi=150, bbox_inches='tight')

print("Grafico guardado en: heatmap_correlaciones.png")
plt.show()


**Conclusion matriz de correlaciones:**
- Area tiene la mayor correlacion con precio (Spearman ~0.7).
- Bathrooms y rooms tambien correlacionan moderadamente con precio.
- No se detecto multicolinealidad severa (|r| >= 0.7) entre predictores.
- Spearman se usa por distribuciones no normales de las variables.


### Ranking de variables por correlacion con precio

In [ ]:
corr_with_price = corr_matrix['precio'].drop('precio').sort_values(ascending=False)
print('--- CORRELACION CON PRECIO (Spearman) ---')
for var, corr_val in corr_with_price.items():
    if pd.notna(corr_val):
        strength = 'fuerte' if abs(corr_val) >= 0.5 else 'moderada' if abs(corr_val) >= 0.3 else 'debil'
        bars = '#' * int(abs(corr_val) * 20)
        print(f'  {var:15s} {corr_val:>+7.4f}  ({strength})  {bars}')
    else:
        print(f'  {var:15s}      NaN  (sin datos)')

print(f'\nVariable con mayor correlacion: {corr_with_price.index[0]} ({corr_with_price.iloc[0]:+.4f})')
print(f'Variable con menor correlacion: {corr_with_price.index[-1]} ({corr_with_price.iloc[-1]:+.4f})')

### Multicolinealidad entre predictores

In [ ]:
predictors = ['area', 'rooms', 'bathrooms', 'parking', 'estrato']
corr_pred = corr_matrix.loc[predictors, predictors]

print('--- MULTICOLINEALIDAD ENTRE PREDICTORES ---')
high_pairs = []
for i in range(len(predictors)):
    for j in range(i+1, len(predictors)):
        val = corr_pred.iloc[i, j]
        if abs(val) >= 0.7:
            high_pairs.append((predictors[i], predictors[j], val))
            print(f'  ALTA: {predictors[i]:15s} vs {predictors[j]:15s}  r={val:.4f}')

if not high_pairs:
    print('  No se detecto multicolinealidad alta (|r| >= 0.7) entre predictores.')
    print('  Las variables predictoras son reasonably independientes.')
else:
    print(f'\nSe detectaron {len(high_pairs)} pares con alta correlacion.')
    print('  Recomendacion: considerar PCA o eliminar una variable del par.')

print('\nCorrelacion completa entre predictores:')
display(corr_pred.round(4))

### Correlaciónes por dataset

In [ ]:
print('--- CORRELACIONES POR DATASET ---\n')
for fid in NUM_CFG.keys():
    sub = df_num[df_num['fuente'] == fid][corr_vars].dropna()
    if len(sub) < 50:
        continue
    cm = sub.corr(method='spearman')
    print(f'{fid} (n={len(sub):,}):')
    print(f'  precio-area:    {cm.loc["precio","area"]:.4f}')
    print(f'  precio-rooms:   {cm.loc["precio","rooms"]:.4f}')
    print(f'  precio-bath:    {cm.loc["precio","bathrooms"]:.4f}')
    print(f'  area-rooms:     {cm.loc["area","rooms"]:.4f}')
    print(f'  area-bath:      {cm.loc["area","bathrooms"]:.4f}')
    print(f'  rooms-bath:     {cm.loc["rooms","bathrooms"]:.4f}')
    print()

---
## Resumen: Variables Categóricas y Matriz de Correlaciones

**Sección 8 - Variables Categoricas:**
- Distribución de tipos de propiedad: Apartamento y Casa dominan (~80%).
- Tipos minoritarios identificados (<1% cada uno).
- Cobertura temporal: la mayoria de datos 2020-2024.
- Cobertura geográfica: Bogota, Medellin concentran el volumen.
- Ciudades intermedias con cobertura insuficiente identificadas.

**Sección 9 - Matriz de Correlaciónes:**
- Correlación con precio: area (mas alta), bathrooms, rooms.
- Sin multicolinealidad severa entre predictores.
- Spearman utilizado por distribuciones no normales.

**Outputs generados:**
- docs/figures/top8_tipos_propiedad.png
- docs/figures/registros_por_year.png
- docs/figures/top12_ciudades_volumen.png
- docs/figures/heatmap_correlaciones.png